In [ ]:
# clone into the repo
# install requirements
# start worker (worker gets gpu status, loads weights to file, gets cloudflare url, registers itself)
# is able to take requests
!git clone https://github.com/prava241/vllm-router.git
%cd vllm-router
!pip install -r requirements.txt

# Download cloudflared into src/worker, since that's where worker.py runs from
%cd src/worker
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
%cd ../..

In [ ]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

In [13]:
# pull latest fixes from the dev branch (push your local changes to dev first)
!git pull origin main

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 468 bytes | 468.00 KiB/s, done.
From https://github.com/prava241/vllm-router
 * branch            main       -> FETCH_HEAD
   e2fe516..5d92dcc  main       -> origin/main
Updating e2fe516..5d92dcc
Fast-forward
 src/worker/worker.py | 3 +++
 1 file changed, 3 insertions(+)


Before running the next cell: on your local machine (next to `server.py`), start the controller and expose it publicly so this Colab worker can reach it:

```
uvicorn src.server:app --host 0.0.0.0 --port 8000
./cloudflared tunnel --url http://localhost:8000
```

Copy the printed `https://*.trycloudflare.com` URL and paste it into `CONTROLLER_URL` below.

In [ ]:
CONTROLLER_URL = "https://budget-committees-facts-montreal.trycloudflare.com"  # replace with your current tunnel URL
!ls

!python -m worker.worker --controller-url {CONTROLLER_URL} --host 0.0.0.0 --port 8000

In [ ]:
!python -c "import torch; print(torch.__version__, torch.version.cuda)"
!pip list | grep -i nvidia
!find / -name "libcudart.so*" 2>/dev/null
